# Production grade QLoRA Training using Llama-2-13B (V2.0)

## Pipeline Stages:
1. **Data Ingestion** → Streaming + Quality Assessment
2. **Data Preprocessing** → Cleaning + Formatting
3. **Data Analysis** → Statistical + Correlation Matrices
4. **Model Training** → QLoRA with Checkpointing
5. **Evaluation** → Perplexity + Generation Quality
6. **RAG Index Building** → Vector Store Creation

**Critical Fixes Applied:**
- Tokenizer: `clean_up_tokenization_spaces=False`
- Metadata Cleaning Pipeline
- Uncertainty Training (15%)
- Modular Architecture
- Comprehensive Metrics

In [1]:
import os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [2]:
# Exact versions for reproducibility
!pip install -q \
    transformers==4.36.0 \
    peft==0.7.1 \
    bitsandbytes==0.41.3 \
    accelerate==0.25.0 \
    datasets==2.16.0 \
    torch==2.1.2 \
    tqdm \
    faiss-cpu \
    sentence-transformers \
    flash-attn --no-build-isolation \
    matplotlib seaborn pandas scikit-learn

import transformers, peft, bitsandbytes, torch
print('\n' + '=' * 60)
print('INFOSAGE AI - TRAINING PIPELINE V2.0')
print('=' * 60)
print(f'transformers: {transformers.__version__}')
print(f'peft:         {peft.__version__}')
print(f'bitsandbytes: {bitsandbytes.__version__}')
print(f'torch:        {torch.__version__}')
print('=' * 60)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

In [4]:
import torch
import time

#Hardware Verification
assert torch.cuda.is_available(), 'FATAL: No CUDA device found'

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print('=' * 60)
print('HARDWARE DIAGNOSTICS')
print('=' * 60)
print(f'  GPU:           {gpu_name}')
print(f'  VRAM:          {vram_gb:.1f} GB')
print(f'  CUDA Version:  {torch.version.cuda}')
print(f'  PyTorch:       {torch.__version__}')
print(f'  BF16 Support:  {torch.cuda.is_bf16_supported()}')

# Enabling TF32 for H100 tensor core acceleration
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.backends.cuda.matmul.fp32_precision = 'tf32'
    torch.backends.cuda.conv.fp32_precision = 'tf32'
    print(f'  TF32:          Enabled (Strict)')
except AttributeError:
    print(f'  TF32:          Enabled (Legacy)')

print('=' * 60)

# 1: Data Ingestion

**Components:**
- Streaming Dataset
- Quality Filtering
- Sampling Strategy

In [5]:
from datasets import load_dataset
from typing import Iterator, Dict, Any
import re

class StreamingDataLoader:
    """Handles streaming data from HuggingFace datasets."""
    
    def __init__(self, dataset_name: str, split: str = 'train'):
        self.dataset_name = dataset_name
        self.split = split
        self.dataset = None
        
    def load(self, shuffle: bool = True, buffer_size: int = 10000):
        """Initialize streaming dataset."""
        print(f'[DataLoader] Loading {self.dataset_name}...')
        self.dataset = load_dataset(
            self.dataset_name,
            split=self.split,
            streaming=True
        )
        
        if shuffle:
            self.dataset = self.dataset.shuffle(seed=42, buffer_size=buffer_size)
        
        print(f'[DataLoader] ✓ Stream initialized')
        return self.dataset

class QualityFilter:
    """Filters low-quality samples based on heuristics."""
    
    def __init__(self, min_length: int = 50, max_length: int = 5000):
        self.min_length = min_length
        self.max_length = max_length
        self.filtered_count = 0
        self.total_count = 0
        
    def is_valid(self, text: str) -> bool:
        """Check if text meets quality criteria."""
        self.total_count += 1
        
        # Length check
        if not (self.min_length <= len(text) <= self.max_length):
            self.filtered_count += 1
            return False
        
        # Ratio checks
        alpha_ratio = sum(c.isalpha() for c in text) / len(text)
        if alpha_ratio < 0.7:  # Too many special chars
            self.filtered_count += 1
            return False
        
        return True
    
    def get_stats(self) -> Dict[str, Any]:
        return {
            'total_processed': self.total_count,
            'filtered_out': self.filtered_count,
            'pass_rate': (self.total_count - self.filtered_count) / max(self.total_count, 1)
        }

class SamplingStrategy:
    """Controls sampling from the dataset stream."""
    
    def __init__(self, max_samples: int = 1_000_000):
        self.max_samples = max_samples
        self.samples_taken = 0
        
    def should_continue(self) -> bool:
        return self.samples_taken < self.max_samples
    
    def record_sample(self):
        self.samples_taken += 1
        
    def progress(self) -> float:
        return self.samples_taken / self.max_samples

print('[Stage 1] Data Ingestion Layer initialized')


# 2: Data Preprocessing Layer

**Components:**
- Metadata Cleaning
- Prompt Formatting  
- Uncertainty Handling

In [6]:
import random

class MetadataCleaner:
    """Removes web scraping artifacts from text."""
    
    PATTERNS = [
        r'\[Reference:.*?\]',
        r'\|answered by\|.*?\|',
        r'\|date created\|.*?\|',
        r'\|last updated\|.*?\|',
        r'\|[Cc]omments\|.*',
        r'answered by:.*?(?=\n|$)',
        r'date created:.*?(?=\n|$)',
        r'last updated:.*?(?=\n|$)',
        r'\bComments:.*?(?=\n|$)',
        r'Source:.*?(?=\n|$)',
        r'Posted by:.*?(?=\n|$)',
        r'\d{1,2}/\d{1,2}/\d{2,4}',
    ]
    
    def __init__(self):
        self.chars_removed_total = 0
        self.samples_cleaned = 0
        
    def clean(self, text: str) -> str:
        """Remove all metadata patterns."""
        original_len = len(text)
        
        for pattern in self.PATTERNS:
            text = re.sub(pattern, '', text, flags=re.IGNORECASE | re.DOTALL)
        
        # Collapse whitespace
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'\s*\|\s*$', '', text)
        
        self.chars_removed_total += (original_len - len(text))
        self.samples_cleaned += 1
        
        return text.strip()

class PromptFormatter:
    """Formats text into Q&A training examples."""
    
    def __init__(self, max_context_len: int = 800):
        self.max_context_len = max_context_len
        
    def format(self, text: str) -> str:
        """Create Q&A format with PROPER SPACING."""
        text = text[:self.max_context_len]
        
        formatted = (
            f"Context: {text}\n\n"
            f"Question: Explain the main topic discussed.\n"
            f"Answer: "  # ← CRITICAL: Space after colon
        )
        return formatted

class UncertaintyInjector:
    """Injects uncertainty examples to teach 'I don't know'."""
    
    TEMPLATES = [
        (
            "Context: Information about photosynthesis in plants.\n\n"
            "Question: How does quantum computing work?\n"
            "Answer: The provided context discusses photosynthesis, not quantum computing. "
            "I don't have enough information to answer this question based on the given context."
        ),
        (
            "Context: Historical facts about World War II.\n\n"
            "Question: What are the latest AI research findings?\n"
            "Answer: The context focuses on World War II history. I cannot answer questions "
            "about recent AI research based on this information."
        ),
        (
            "Context: Basic mathematics concepts.\n\n"
            "Question: Explain the biochemical pathway of glycolysis.\n"
            "Answer: The context covers mathematics, not biochemistry. I don't have the necessary "
            "information to explain glycolysis from this context."
        ),
        (
            "Context: Description of solar system planets.\n\n"
            "Question: How does cryptocurrency blockchain work?\n"
            "Answer: The context is about astronomy, not cryptocurrency technology. "
            "I cannot provide information on blockchain from this context."
        ),
    ]
    
    def __init__(self, injection_rate: float = 0.15):
        self.injection_rate = injection_rate
        self.injected_count = 0
        
    def should_inject(self) -> bool:
        """Decide if this sample should be an uncertainty example."""
        return random.random() < self.injection_rate
    
    def get_example(self) -> str:
        """Return random uncertainty template."""
        self.injected_count += 1
        return random.choice(self.TEMPLATES)

print('[Stage 2] Preprocessing Layer initialized')


# 3: Data Analysis Layer

**Components:**
- Statistical Metrics
- Correlation Analysis
- Visualization Generator

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm

# Initialize pipeline components
loader = StreamingDataLoader('HuggingFaceFW/fineweb-edu')
quality_filter = QualityFilter()
metadata_cleaner = MetadataCleaner()

# Load dataset
raw_dataset = loader.load()

# Sample for analysis
ANALYSIS_SAMPLES = 5000
sample_data = []

print(f'\n[Analysis] Sampling {ANALYSIS_SAMPLES:,} examples for quality assessment...')
for i, row in enumerate(tqdm(raw_dataset, total=ANALYSIS_SAMPLES, desc='Sampling')):
    if i >= ANALYSIS_SAMPLES:
        break
    
    text = row['text']
    is_valid = quality_filter.is_valid(text)
    cleaned = metadata_cleaner.clean(text)
    
    sample_data.append({
        'original_length': len(text),
        'cleaned_length': len(cleaned),
        'chars_removed': len(text) - len(cleaned),
        'has_metadata': int(len(text) > len(cleaned)),
        'word_count': len(text.split()),
        'avg_word_length': np.mean([len(w) for w in text.split()]) if text.split() else 0,
        'quality_pass': int(is_valid),
        'alpha_ratio': sum(c.isalpha() for c in text) / max(len(text), 1),
    })

df = pd.DataFrame(sample_data)

print('\n' + '=' * 60)
print('DATA QUALITY REPORT')
print('=' * 60)
print(df.describe())
print('\n' + '=' * 60)
print(f'Quality pass rate:      {100 * df["quality_pass"].mean():.1f}%')
print(f'Metadata found in:      {df["has_metadata"].sum():,} / {len(df):,} '
      f'({100 * df["has_metadata"].mean():.1f}%)')
print(f'Avg chars removed:      {df["chars_removed"].mean():.0f}')
print(f'Median text length:     {df["cleaned_length"].median():.0f} chars')
print('=' * 60)

In [8]:
# Correlation Analysis
print('\n[Analysis] Generating correlation matrix...')

correlation_features = [
    'original_length', 'cleaned_length', 'chars_removed',
    'word_count', 'avg_word_length', 'alpha_ratio'
]

corr_matrix = df[correlation_features].corr()

print('\nCORRELATION MATRIX:')
print('=' * 80)
print(corr_matrix.round(3))
print('=' * 80)

# Key insights
print('\nKEY INSIGHTS:')
print('  • original_length ↔ cleaned_length:', f"{corr_matrix.loc['original_length', 'cleaned_length']:.3f}")
print('  • word_count ↔ cleaned_length:', f"{corr_matrix.loc['word_count', 'cleaned_length']:.3f}")
print('  • chars_removed ↔ original_length:', f"{corr_matrix.loc['chars_removed', 'original_length']:.3f}")

In [ ]:
# Comprehensive Visualization
print('\n[Analysis] Generating visualizations...')

fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Length Distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df['cleaned_length'], bins=50, edgecolor='black', alpha=0.7, color='#3b82f6')
ax1.axvline(df['cleaned_length'].median(), color='red', linestyle='--', linewidth=2, label='Median')
ax1.set_title('Text Length Distribution', fontsize=12, fontweight='bold')
ax1.set_xlabel('Character Count')
ax1.set_ylabel('Frequency')
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Metadata Impact
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(df['chars_removed'], bins=50, edgecolor='black', alpha=0.7, color='#f59e0b')
ax2.set_title('Metadata Removal Impact', fontsize=12, fontweight='bold')
ax2.set_xlabel('Characters Removed')
ax2.set_ylabel('Frequency')
ax2.grid(alpha=0.3)

# 3. Quality Pass Rate
ax3 = fig.add_subplot(gs[0, 2])
quality_counts = df['quality_pass'].value_counts()
ax3.bar(['Failed', 'Passed'], 
        [quality_counts.get(0, 0), quality_counts.get(1, 0)],
        color=['#ef4444', '#10b981'], alpha=0.7, edgecolor='black')
ax3.set_title('Quality Filter Pass Rate', fontsize=12, fontweight='bold')
ax3.set_ylabel('Sample Count')
ax3.grid(alpha=0.3)

# 4. Correlation Heatmap
ax4 = fig.add_subplot(gs[1, :])
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax4, 
            square=True, cbar_kws={'label': 'Correlation Coefficient'})
ax4.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')

# 5. Word Count Distribution
ax5 = fig.add_subplot(gs[2, 0])
ax5.hist(df['word_count'], bins=50, edgecolor='black', alpha=0.7, color='#8b5cf6')
ax5.set_title('Word Count Distribution', fontsize=12, fontweight='bold')
ax5.set_xlabel('Word Count')
ax5.set_ylabel('Frequency')
ax5.grid(alpha=0.3)

# 6. Alpha Ratio
ax6 = fig.add_subplot(gs[2, 1])
ax6.hist(df['alpha_ratio'], bins=50, edgecolor='black', alpha=0.7, color='#ec4899')
ax6.set_title('Alphabetic Character Ratio', fontsize=12, fontweight='bold')
ax6.set_xlabel('Alpha Ratio')
ax6.set_ylabel('Frequency')
ax6.grid(alpha=0.3)

# 7. Metadata Presence
ax7 = fig.add_subplot(gs[2, 2])
metadata_counts = df['has_metadata'].value_counts()
ax7.pie([metadata_counts.get(0, 0), metadata_counts.get(1, 0)],
        labels=['Clean', 'Has Metadata'],
        autopct='%1.1f%%',
        colors=['#10b981', '#ef4444'],
        startangle=90)
ax7.set_title('Metadata Presence', fontsize=12, fontweight='bold')

plt.suptitle('InfoSage AI - Data Quality Analysis', fontsize=16, fontweight='bold', y=0.995)

save_path = '/content/drive/MyDrive/fineweb_edu_llama2_13b/data_analysis.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f'\n✓ Visualization saved: {save_path}')
plt.show()

# Export statistics
stats_dict = {
    'total_samples': len(df),
    'quality_pass_rate': float(df['quality_pass'].mean()),
    'metadata_rate': float(df['has_metadata'].mean()),
    'avg_length': float(df['cleaned_length'].mean()),
    'median_length': float(df['cleaned_length'].median()),
    'avg_words': float(df['word_count'].mean()),
    'correlation_matrix': corr_matrix.to_dict()
}

import json
stats_path = '/content/drive/MyDrive/fineweb_edu_llama2_13b/data_stats.json'
with open(stats_path, 'w') as f:
    json.dump(stats_dict, f, indent=2)

print(f'✓ Statistics exported: {stats_path}')


# 4: Model Training

**Components:**
- QLoRA Setup
- Training Loop
- Checkpoint Manager

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL_NAME = 'NousResearch/Llama-2-13b-hf'
MAX_LENGTH = 1024

print(f'\n[Stage 4] Loading {MODEL_NAME}...')

# CRITICAL FIX: Proper tokenizer settings
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    clean_up_tokenization_spaces=False,  # ← Preserves ▁ markers
    use_fast=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = MAX_LENGTH

# Verify tokenization
test_tokens = tokenizer.tokenize("The model outputs text")
if any('▁' in t for t in test_tokens):
    print('Tokenization preserves SentencePiece markers')
else:
    print('WARNING: Spacing markers may be missing')

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2'
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()

print('Base model loaded')

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']
)

model = get_peft_model(model, peft_config)

trainable, total = 0, 0
for p in model.parameters():
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()

print('\n' + '=' * 60)
print('LORA CONFIGURATION')
print('=' * 60)
print(f'  Total params:     {total:,}')
print(f'  Trainable params: {trainable:,}')
print(f'  Trainable %:      {100 * trainable / total:.2f}%')
print('=' * 60)

In [ ]:
print('\nPreparing training data stream...')

# Reload dataset for training
loader = StreamingDataLoader('HuggingFaceFW/fineweb-edu')
raw_dataset = loader.load()

# Initialize preprocessing components
cleaner = MetadataCleaner()
formatter = PromptFormatter()
uncertainty = UncertaintyInjector(injection_rate=0.15)

def tokenize_with_pipeline(examples):
    """Full preprocessing pipeline."""
    processed_texts = []
    
    for text in examples['text']:
        # 15% uncertainty injection
        if uncertainty.should_inject():
            processed_texts.append(uncertainty.get_example())
        else:
            # Clean → Format → Add
            cleaned = cleaner.clean(text)
            if len(cleaned) > 50:
                formatted = formatter.format(cleaned)
                processed_texts.append(formatted)
            else:
                # Fallback to uncertainty if too short
                processed_texts.append(uncertainty.get_example())
    
    return tokenizer(
        processed_texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

# Detect columns to remove
try:
    sample = next(iter(raw_dataset))
    remove_cols = list(sample.keys())
except:
    remove_cols = ['text', 'id', 'url', 'dump', 'segment', 'timestamp']

tokenized_dataset = raw_dataset.map(
    tokenize_with_pipeline,
    batched=True,
    batch_size=32,
    remove_columns=remove_cols
)

print('Training pipeline configured:')
print('   Metadata cleaning: ENABLED')
print('   Prompt formatting: ENABLED')
print('   Uncertainty injection: 15%')
print('   SentencePiece preservation: ENABLED')

In [ ]:
output_dir = '/content/drive/MyDrive/fineweb_edu_llama2_13b/checkpoints'
os.makedirs(output_dir, exist_ok=True)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_steps=150,
    max_steps=5000,
    bf16=True,
    optim='adamw_bnb_8bit',
    gradient_checkpointing=True,
    logging_steps=20,
    save_steps=500,
    save_total_limit=3,
    report_to='none',
    remove_unused_columns=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True
)

print('Training configuration ready')

In [ ]:
from transformers import TrainerCallback
import gc

class MemoryCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    callbacks=[MemoryCallback()]
)

print('✓ Trainer initialized')

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(output_dir)
train_start = time.time()

print('\n' + '=' * 60)
print('STARTING TRAINING')
print('=' * 60)

try:
    if last_checkpoint:
        print(f'Resuming from: {last_checkpoint}')
        trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        print('Starting from scratch...')
        trainer.train()
    
    elapsed = (time.time() - train_start) / 60
    print('\n' + '=' * 60)
    print(f'✓ TRAINING COMPLETE: {elapsed:.1f} minutes')
    print('=' * 60)
    
except Exception as e:
    print(f'\n❌ Training failed: {e}')
    torch.cuda.empty_cache()


# 5: Model Evaluating

**Components:**
- Generation Quality Tests
- Spacing Verification
- Uncertainty Response Check

In [ ]:
print('\nEvaluating model quality...')

# Test prompts
test_cases = [
    {
        'name': 'Basic Knowledge',
        'prompt': 'Context: Photosynthesis is the process by which plants convert sunlight into energy.\n\nQuestion: What is photosynthesis?\nAnswer:',
        'expected': 'should mention sunlight and energy'
    },
    {
        'name': 'Uncertainty Test',
        'prompt': 'Context: Information about the solar system.\n\nQuestion: How does blockchain work?\nAnswer:',
        'expected': 'should express uncertainty or lack of context'
    },
]

print('\nGENERATION QUALITY TESTS:')
print('=' * 60)

for test in test_cases:
    inputs = tokenizer(test['prompt'], return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.5,
            do_sample=True
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    # Check for spacing issues
    has_spacing_issues = bool(re.search(r'[a-z]{15,}', response))
    spacing_status = 'SPACING ISSUES' if has_spacing_issues else 'Clean'
    
    print(f'\nTest: {test["name"]}')
    print(f'Expected: {test["expected"]}')
    print(f'Response: {response[:150]}...')
    print(f'Spacing: {spacing_status}')
    print('-' * 60)

print('\nEvaluation complete')

In [ ]:
final_model_dir = '/content/drive/MyDrive/fineweb_edu_llama2_13b/final_model'
print(f'\nSaving model to: {final_model_dir}')
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

# Save training metadata
metadata = {
    'architecture': 'Modular ML Pipeline',
    'model': MODEL_NAME,
    'versions': {
        'transformers': transformers.__version__,
        'peft': peft.__version__,
        'torch': torch.__version__,
    },
    'pipeline_components': {
        'data_ingestion': ['StreamingDataLoader', 'QualityFilter', 'SamplingStrategy'],
        'preprocessing': ['MetadataCleaner', 'PromptFormatter', 'UncertaintyInjector'],
        'analysis': ['Statistical', 'Correlation', 'Visualization'],
        'training': ['QLoRA', 'Checkpointing'],
        'evaluation': ['GenerationQuality', 'SpacingVerification'],
        'rag': ['EmbeddingGenerator', 'FAISSIndex']
    },
    'preprocessing_stats': {
        'samples_cleaned': cleaner.samples_cleaned,
        'chars_removed': cleaner.chars_removed_total,
        'uncertainty_injected': uncertainty.injected_count,
        'uncertainty_rate': uncertainty.injection_rate
    },
    'training_config': {
        'max_length': MAX_LENGTH,
        'max_steps': 5000,
        'batch_size': 2,
        'grad_accumulation': 4,
        'learning_rate': 1e-4
    },
    'tokenizer': {
        'clean_up_tokenization_spaces': False,
        'preserves_sentencepiece': True
    }
}

with open(f'{final_model_dir}/training_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Model and metadata saved')


# 6: Building Rag Index

**Components:**
- Embedding Generator
- FAISS Index
- Export Pipeline

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np

print('\n[Stage 6] Building RAG index...')

RAG_SAMPLES = 100_000
RAG_DIR = '/content/drive/MyDrive/fineweb_edu_llama2_13b/rag_index'
os.makedirs(RAG_DIR, exist_ok=True)

# Reload dataset
rag_loader = StreamingDataLoader('HuggingFaceFW/fineweb-edu')
rag_dataset = rag_loader.load(shuffle=False)  # No shuffle for RAG
rag_stream = rag_dataset.take(RAG_SAMPLES)

# Initialize cleaner for RAG
rag_cleaner = MetadataCleaner()

passages = []
print(f'Extracting and cleaning {RAG_SAMPLES:,} passages...')
for row in tqdm(rag_stream, total=RAG_SAMPLES, desc='Processing'):
    text = row['text'].strip()
    
    # CRITICAL: Clean before chunking
    text = rag_cleaner.clean(text)
    
    # Chunk into passages
    for i in range(0, len(text), 500):
        chunk = text[i:i + 500].strip()
        if len(chunk) > 50:
            passages.append(chunk)

print(f'\nEncoding {len(passages):,} passages...')
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = embedder.encode(
    passages,
    show_progress_bar=True,
    batch_size=256,
    convert_to_numpy=True
)

# Build FAISS index
print('Building FAISS index...')
index = faiss.IndexFlatIP(embeddings.shape[1])
faiss.normalize_L2(embeddings)
index.add(embeddings)

# Save
faiss.write_index(index, os.path.join(RAG_DIR, 'faiss_index.bin'))
np.save(os.path.join(RAG_DIR, 'passages.npy'), np.array(passages, dtype=object))

print('\n' + '=' * 60)
print('RAG INDEX COMPLETE')
print('=' * 60)
print(f'  Passages: {len(passages):,}')
print(f'  Dimension: {embeddings.shape[1]}')
print(f'  Location: {RAG_DIR}')
print('=' * 60)